- a) Write a Python program that uses Least Squares analysis to fit a 4th order interpolating polynomial
through the data.
    * In the report, present the objective function plus the resulting set of equations that will
be solved by your Python program.
    * Your Python code needs to be included in your report.
    * The Python program should have a dedicated function to read the data file and then to
construct the system of equations.
    * The linear equations can be solved using one of the routines you wrote in question 1.
    * Report the final value of the solution plus the final order vector.
- b) Using the interpolating polynomial that you created in part a) estimate the price of silver in July
26, 1969; May 17, 1992 and January 31, 2026 (yes, I am asking you to extrapolate). Report the
computed values.
- c) Comment on any data cleaning that you did in order to ensure that the data was usable with your
program.
- d) Write a Python program that uses Least Squares analysis to fit a 5th order interpolating polynomial
through the data.
    * In the report, present the objective function plus the resulting set of equations that will
be solved by your Python program.
    * Your Python code needs to be included in your report.
    * If you were able to write a single program that can fit both the 4th and the 5 th order
polynomials, you can simply reference the program that you wrote for part a).
    * The Python program should have a dedicated function to read the data file and then to
construct the system of equations.
    * The linear equations can be solved using one of the routines you wrote in question 1.
    * Report the final value of the solution plus the final order vector.
- e) Using the interpolating polynomial that you created in part a) estimate the price of silver in July
26, 1969; May 17, 1992 and January 31, 2026. Report the computed values.
- f) Compare the computed values from part b) and d) and comment on any discrepancies.

In [ ]:
# Rough order of things:
import csv
from helper_functions.matrix_solving import gauss_elim_p_piv
'''
    1. Create file reading function (with open ___ as ___:)
        Args:
            n: number of order desired
            resource: path to the CSV file being used
        
        Returns:
            list of the resultant coefficients.
        
        - Reads files
        - collects data
        - constructs equations
    
    2. Create function to go through and take calculated coefficients and x value,
        and return a y value.
        Args:
            x: desired x value
            file: the CSV file to read from
            n: order of fit desired
        
        Returns:
            Y: The resultant value from the calculation
    
    NOTE: Function 2 should call function 1, so it can be nice and small.

    3. Potential helper function:
        takes a date, and converts it into a useable X value.
        this would allow the x value input to be either a pre calculated float, or a date in
        format given in question (Ex. April 26, 1986)
'''

def DateToNum(date):
    month_day,year = date.split(',')
    month,day = month_day.split(' ')
    month = month.lower().strip()
    day = int(day.strip())
    year = int(year.strip())
    months = ['january','february','march','april',
                  'may','june','july','august','september','october',
                  'november','december']
    nums = [31,28,31,30,31,30,31,31,30,31,30,31]
    for i in range(0,months.index(month)):
        day+=nums[i]
    day /= 365
    year+=day
    return(round(year,3))


def GetEquation(n:int,resource:str)->tuple:
    '''
    takes in a file and order level, and returns an (n+1)*(n+1) matrix
    Args:
        n:int
    '''
    with open(resource,'r') as file:
        reader = csv.reader(file)
        years = []
        price_sum = []
        for year,price in reader: 
            years.append(int(year)-1984)
            price_sum.append(float(price))
    
    
    matrix = []
    ai = []
    outs = []
    for i in range(n,-1,-1):
        xi = sum(((k**i)*r)for k,r in zip(years,price_sum))
        outs.append(xi)
        row_exp=[]
        ai.append(f'a{i}')
        for j in range(n,-1,-1):
            sum_num = sum(k**(i+j) for k in years)
            row_exp.append(sum_num)
        matrix.append(row_exp)

    results, order= gauss_elim_p_piv(matrix,ai,outs)
    return(results, order)

def approx_price(x,order:int,file:str):
    if type(x)==str:
        x = DateToNum(x) - 1984
    else:
        x -= 1984
    
    y = 0
    eqtn_vars, _ = GetEquation(order,file)
    for i in range(order,-1,-1):
        y+= (eqtn_vars[f'a{i}']*(x**i))
    return(y)

#TODO:

if __name__ == '__main__':
    """ 
    After slight experimentation, most accurate order depends on the date provided
    ex: for April 26, 1989 most accurate order was 6th, while for the year 1987, the most accurate
    order is the 5th, therefore it depends. The only solid conclusion is that 4th is way too far off
    to be accurate.
    """
    # Part a.
    date_str = 'April 26, 1989'
    file = 'Silver_price.csv'
    q2_p1_a,q2_p1_b = GetEquation(4,file)
    print(f'Order: {q2_p1_b},\nCoeffs: {q2_p1_a}')

    # Part b.
    print('\n=====','4th order','=====')
    dates = ['July 26, 1969','May 17, 1992','January 31, 2026']
    for date in dates:
        y = approx_price(date,4,file)
        print(f'Approximated price for {date}: ${round(y,2)}')

    # Part e.
    print('\n=====','5th order','=====')
    for date in dates:
        y = approx_price(date,5,file)
        print(f'Approximated price for {date}: ${round(y,2)}')

    



Order: [0, 1, 2, 3, 4],
Coeffs: {'a4': 0.0002138590865010927, 'a3': -0.018564080038763326, 'a2': 0.5506350286739695, 'a1': -5.644726729582376, 'a0': 25.99058676467693}

===== 4th order =====
Approximated price for July 26, 1969: $287.26
Approximated price for May 17, 1992: $7.48
Approximated price for January 31, 2026: $50.81

===== 5th order =====
Approximated price for July 26, 1969: $-806.57
Approximated price for May 17, 1992: $9.2
Approximated price for January 31, 2026: $76.41
